In [19]:
import pandas as pd
import numpy as np
import re

from extraction.texts import model_loader, text_encoding

In [48]:
df = pd.read_csv("data/texts/cyberbullying.csv")

category = {i: (1 if i != 'not_cyberbullying' else 0) for i in df["cyberbullying_type"].unique()}
df["cyberbullying_type"] = df["cyberbullying_type"].replace(category)

df = pd.concat([
    df[df["cyberbullying_type"] == 0].head(7500),
    df[df["cyberbullying_type"] == 1].head(30000)
]).reset_index(drop=True)

df["tweet_text"] = (
    df["tweet_text"].apply(lambda x: re.sub(r"[^A-Za-z0-9]", " ", 
                                             x, count=0, flags=0))
            .apply(lambda x: re.findall(r"[A-Za-z0-9]+", x))
            .apply(lambda x: " ".join(x))
)

/var/folders/tg/66tmmx452mg3qln33ts5xqh80000gn/T/ipykernel_23204/2272108052.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["cyberbullying_type"] = df["cyberbullying_type"].replace(category)


In [49]:
texts = df["tweet_text"].to_list()

In [50]:
model, model_code = model_loader('mps', model_code='e5')
text_vectors = text_encoding(texts, model, model_code)

Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

In [51]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [52]:
x, y = text_vectors, df["cyberbullying_type"].to_numpy()
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    stratify=y,
                                                    test_size=0.2)

In [60]:
log_reg = LogisticRegression(
    penalty='l1',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='saga',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.31,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9083 (When flagged positive, accuracy is 90.83%)
Custom Recall Score:    0.8588 (Captured 85.88% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9107 (When flagged positive, accuracy is 91.07%)
Custom Recall Score:    0.8538 (Captured 85.38% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9027 (When flagged positive, accuracy is 90.27%)
Custom Recall Score:    0.8429 (Captured 84.29% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9019 (When flagged positive, accuracy is 90.19%)
Custom Recall Score:    0.8431 (Captured 84.31% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9023 (When flagged positive, accuracy is 90.23%)
Custom R

In [57]:
log_reg = LogisticRegression(
    penalty='l1',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='saga',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.31

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

           0       0.55      0.59      0.57      1500
           1       0.90      0.88      0.89      6000

    accuracy                           0.82      7500
   macro avg       0.72      0.74      0.73      7500
weighted avg       0.83      0.82      0.82      7500

[[ 890  610]
 [ 734 5266]]


In [58]:
import joblib

In [59]:
joblib.dump(log_reg, open("model/cyberbullying.jobllib", 'wb'))